# LightGBM LambdaRank · Adri (rol B)

Objetivo: mejorar el LightGBM de referencia (CV ≈ 0.651 con las columnas originales).

**Reglas (EQUIPO.md):**
- Cada snapshot se predice **solo con sus propias filas**: toda feature nueva se calcula con `groupby("sample_id")` (o un grupo más fino dentro del snapshot).
- Validación: los 5 folds de `Data/folds.csv` (por controlador). Se decide por la CV, no por Kaggle.
- Una feature solo entra si **mejora la CV**.

**Cómo usar el notebook:** ejecuta de arriba abajo (**Run All**). Con `RAPIDO = True` (sección 1) tarda ~8–10 min; con `False`, ~30 min. La sección 5 hace el barrido y elige sola la mejor combinación; la sección 7 entrena el modelo final con ella y guarda los archivos. Cada experimento se apunta en `experiments/lgb_runs.csv`.

## 1. Configuración

In [1]:
import json, subprocess, sys, time
import numpy as np, pandas as pd, lightgbm as lgb
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "Data"
for p in ("oof", "preds", "submissions", "models", "experiments"):
    (ROOT / p).mkdir(exist_ok=True)

NAME = "lgb_adri"   # nombre de los archivos de salida
TYPES = ['entry_conflict_resolution', 'exit_conflict_resolution', 'entry_coordination', 'exit_coordination',
         'assume_aircraft', 'transfer_aircraft', 'conflict_resolution', 'non_conformance_resolution',
         'qos_improvement', 'entry_conditions', 'exit_conditions', 'aircraft_request', 'zone_conflict',
         'return_to_route']

# Parámetros base de EQUIPO.md
PARAMS = dict(objective="lambdarank", n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=50,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.7, lambdarank_truncation_level=5,
              random_state=0, verbose=-1)
# RAPIDO = True: ~8-10 min (1 semilla, 4 grupos candidatos, sin barrido de parámetros, final con 2 semillas)
# RAPIDO = False: ~30 min (2 semillas, 6 grupos, barrido de parámetros, final con 3 semillas)
RAPIDO = True
SEEDS_BARRIDO = [0] if RAPIDO else [0, 1]       # semillas por experimento del barrido (se promedian)
SEEDS_FINAL = [0, 1] if RAPIDO else [0, 1, 2]   # semillas del modelo final

## 2. Datos
- `lab` = train + validation (con etiqueta), cada fila con su `fold` (0–4).
- `test` = test, sin etiqueta.
- `cs` = callsign del avión de la tarea (sacado de `task_name`). **No es una feature**: solo sirve para agrupar las tareas de un mismo avión.

In [2]:
def load(name):
    d = pd.read_csv(DATA / f"{name}.csv")
    pref = d.task_index.map(lambda t: len(TYPES[t]) + 1)
    d["cs"] = [n[p:].split("_")[0] for n, p in zip(d.task_name, pref)]
    return d

lab = pd.concat([load("train"), load("validation")], ignore_index=True)
lab = lab.merge(pd.read_csv(DATA / "folds.csv"), on="atco", how="left")
test = load("test")
lab = lab.sort_values(["sample_id", "row_index"]).reset_index(drop=True)
test = test.sort_values(["sample_id", "row_index"]).reset_index(drop=True)

assert lab.fold.notna().all(), "hay controladores sin fold"
assert (lab.groupby("sample_id").target_index.sum() == 1).all(), "alguna muestra no tiene exactamente 1 respuesta"
print(f"lab: {lab.sample_id.nunique():,} muestras, {len(lab):,} filas | test: {test.sample_id.nunique():,} muestras")
print(lab.drop_duplicates("sample_id").groupby("fold").atco.unique())

lab: 13,175 muestras, 474,688 filas | test: 1,863 muestras
fold
0        [7, 14, 18]
1      [1, 3, 4, 15]
2      [2, 6, 9, 16]
3        [8, 17, 19]
4    [5, 10, 11, 20]
Name: atco, dtype: object


## 3. Features
Cada grupo es una función que recibe el dataframe y devuelve **solo las columnas nuevas**. Todas agrupan por `sample_id` (o por avión / tipo dentro del snapshot), así que solo usan el propio snapshot.

| Grupo | Origen | Qué añade |
|---|---|---|
| `snapshot` | EQUIPO.md | Ranking y diferencia respecto al mínimo **dentro del snapshot** de columnas de urgencia/actividad |
| `tipo` | EQUIPO.md | Los mismos rankings **dentro de las tareas del mismo tipo** |
| `avion` | EQUIPO.md | Si la tarea es la más antigua de su avión, cuántos tipos distintos tiene el avión |
| `tipo_estado` | EQUIPO.md + Pau #2 | Cruces tipo × estado: assume con oferta, transfer debido, **entry_coordination con gemelo (`ts_twin_entry` = `is_entry_losing`)**, zona con su distancia… |
| `contexto` | EQUIPO.md | Cómo está el snapshot: nº de aviones, transfers/assumes/zonas/conflictos pendientes |
| `exit_coord` | **Pau #3** | exit_coordination × estado de la coordinación con el sector siguiente (`k_out_*`, mismatch, herramienta), si el avión también tiene transfer, y edad de la tarea frente a las otras de su avión |
| `alerta` | **Pau #4** | Conflictos con alerta STCA en pantalla, nº de alertas del snapshot, ranking del tiempo al mínimo entre las tareas de conflicto |
| `deadline` | nueva | Un "tiempo límite" común para todos los tipos (entrada: tiempo al sector; salida: tiempo al punto de salida; conflicto: tiempo al mínimo; zona: tiempo hasta la zona) y su ranking en el snapshot y en el tipo |

Pau #5 (ranking de distancia al borde dentro del tipo) ya es una columna original (`l_rank_absf1_type`). Pau #1 (oráculo) se descarta: usa `scenario` y `timestamp`, prohibidos como features.

In [3]:
META = ['sample_id', 'target_index', 'session', 'atco', 'scenario', 'timestamp', 'task_name', 'row_index',
        'fold', 'cs']
BASE = [c for c in lab.columns if c not in META]
RANK_COLS = ["a_since_flight", "x_task_age", "x_state_age", "time_to_MD", "k_timeToExitPoint",
             "distance_to_zone", "abs_tts"]
ENTRY_TYPES, EXIT_TYPES, CONFLICT_TYPES = [0, 2, 4, 9], [1, 3, 5, 10], [0, 1, 6]


def _with_abs(df):
    return df.assign(abs_tts=df.time_to_sector.abs())


def _ac_any(df, mask):
    """1 si alguna tarea del mismo avión (en el mismo snapshot) cumple mask."""
    return mask.astype(int).groupby([df.sample_id, df.cs]).transform("max")


def fe_snapshot(df):
    x = _with_abs(df)
    g = x.groupby("sample_id")
    out = {}
    for c in RANK_COLS:
        out[f"rs_{c}"] = g[c].rank(method="min")
        out[f"ds_{c}"] = x[c] - g[c].transform("min")
    return pd.DataFrame(out, index=df.index)


def fe_tipo(df):
    x = _with_abs(df)
    g = x.groupby(["sample_id", "task_index"])
    out = {}
    for c in RANK_COLS:
        out[f"rt_{c}"] = g[c].rank(method="min")
        out[f"dt_{c}"] = x[c] - g[c].transform("min")
    return pd.DataFrame(out, index=df.index)


def fe_avion(df):
    g = df.groupby(["sample_id", "cs"])
    out = {
        "ac_age_rank": g.x_task_age.rank(method="min", ascending=False),   # 1 = la tarea más antigua del avión
        "ac_n_types": g.task_index.transform("nunique"),
        "ac_has_assume": _ac_any(df, df.task_index == 4),
    }
    return pd.DataFrame(out, index=df.index)


def fe_tipo_estado(df):
    t = df.task_index
    out = {
        "ts_assume_offered": ((t == 4) & (df.x_transfer_in == 1)).astype(int),
        "ts_transfer_due": ((t == 5) & (df.k_aircraftNeedsToBeTransferred == 1)).astype(int),
        "ts_entry_coord_done": ((t == 2) & (df.x_isc_coordinated == 1)).astype(int),
        "ts_entry_notified": ((t == 2) & (df.x_isc_notified == 1)).astype(int),
        "ts_zone_dist": df.distance_to_zone.where(t == 12),
        "ts_conflict_ttmd": df.time_to_MD.where(t.isin(CONFLICT_TYPES)),
        "ts_twin_assume": ((t == 4) & (df.l_twin_AE == 1)).astype(int),
        "ts_twin_entry": ((t == 2) & (df.l_twin_AE == 1)).astype(int),     # = is_entry_losing de Pau
    }
    return pd.DataFrame(out, index=df.index)


def fe_contexto(df):
    g = df.groupby("sample_id")
    tmp = pd.DataFrame({
        "sample_id": df.sample_id,
        "is_assume_offer": ((df.task_index == 4) & (df.x_transfer_in == 1)).astype(int),
        "is_transfer_due": ((df.task_index == 5) & (df.k_aircraftNeedsToBeTransferred == 1)).astype(int),
        "is_zone": (df.task_index == 12).astype(int),
        "is_conflict": df.task_index.isin(CONFLICT_TYPES).astype(int),
        "is_entry_coord": (df.task_index == 2).astype(int),
    })
    gt = tmp.groupby("sample_id")
    out = {"cx_n_aircraft": g.cs.transform("nunique")}
    for c in tmp.columns[1:]:
        out["cx_n_" + c[3:]] = gt[c].transform("sum")
    return pd.DataFrame(out, index=df.index)


def fe_exit_coord(df):
    t = df.task_index
    ec = t == 3
    out = {
        "ec_notified": (ec & (df.k_out_NOTIFIED == 1)).astype(int),
        "ec_coordinated": (ec & (df.k_out_COORDINATED == 1)).astype(int),
        "ec_mismatch": (ec & (df.k_outCoordMismatch == 1)).astype(int),
        "ec_tool": (ec & (df.k_coordinationToolActivation == 1)).astype(int),
        "ec_revision": df.k_outCoordRevision.where(ec),
        "ec_with_transfer": (ec & (_ac_any(df, t == 5) == 1)).astype(int),
        "tr_with_exit_coord": ((t == 5) & (_ac_any(df, ec) == 1)).astype(int),
        # 0 = la tarea más nueva de su avión; positivo = lleva más tiempo abierta que la más nueva
        "age_vs_newest_ac": df.x_task_age - df.groupby(["sample_id", "cs"]).x_task_age.transform("min"),
    }
    return pd.DataFrame(out, index=df.index)


def fe_alerta(df):
    conf = df.task_index.isin(CONFLICT_TYPES)
    stca = df.HMI_conflict_safety_alert == 1
    ttmd = df.time_to_MD.where(conf)
    out = {
        "al_conflict_stca": (conf & stca).astype(int),
        "al_n_stca": stca.astype(int).groupby(df.sample_id).transform("sum"),
        "al_ttmd_rank": ttmd.groupby(df.sample_id).rank(method="min"),
        "al_conflict_multi": (conf & (df.multiple_conflicts_ind == 1)).astype(int),
        "al_conflict_dmd": df.distance_to_MD.where(conf),
    }
    return pd.DataFrame(out, index=df.index)


def fe_deadline(df):
    t = df.task_index
    dl = pd.Series(np.nan, index=df.index)
    entry, exit_, zone = t.isin(ENTRY_TYPES), t.isin(EXIT_TYPES), t == 12
    dl[entry] = df.time_to_sector[entry]
    dl[exit_] = df.k_timeToExitPoint.fillna(df.time_to_sector.abs())[exit_]
    dl[t == 6] = df.time_to_MD[t == 6]
    dl[zone] = (df.distance_to_zone * 1852 / df.k_groundSpeed.fillna(230))[zone]   # NM -> m, m/s
    out = {
        "dl": dl,
        "dl_rank": dl.groupby(df.sample_id).rank(method="min"),
        "dl_rank_type": dl.groupby([df.sample_id, t]).rank(method="min"),
        "dl_minus_min": dl - dl.groupby(df.sample_id).transform("min"),
    }
    return pd.DataFrame(out, index=df.index)


FEATURE_GROUPS = {"snapshot": fe_snapshot, "tipo": fe_tipo, "avion": fe_avion,
                  "tipo_estado": fe_tipo_estado, "contexto": fe_contexto,
                  "exit_coord": fe_exit_coord, "alerta": fe_alerta, "deadline": fe_deadline}


def build(df, groups):
    """Devuelve (df con las columnas nuevas, lista de features a usar)."""
    new = [FEATURE_GROUPS[g](df) for g in groups]
    out = pd.concat([df] + new, axis=1)
    return out, BASE + [c for n in new for c in n.columns]

## 4. Entrenamiento y CV
- `fit`: entrena un `LGBMRanker`. Le dice a LightGBM qué filas forman cada snapshot (`group`) para que aprenda a **ordenar dentro del snapshot**.
- `hit3`: para cada snapshot, 1 si la respuesta está entre las 3 de mayor score.
- `cv`: para cada fold, entrena con los otros 4 y predice ese. Con varias semillas entrena un modelo por semilla y **promedia** sus predicciones: la nota es más estable y el modelo algo mejor. El resultado es el **OOF** (out-of-fold).

In [4]:
def fit(A, F, params):
    m = lgb.LGBMRanker(**params)
    m.fit(A[F], A.target_index, group=A.groupby("sample_id", sort=False).size().values,
          categorical_feature=["task_index"])
    return m


def hit3(d, score):
    t = d[["sample_id", "target_index"]].assign(s=np.asarray(score))
    t = t.sort_values(["sample_id", "s"], ascending=[True, False])
    return t.groupby("sample_id").head(3).groupby("sample_id").target_index.max()


def cv(d, F, params, seeds=(0,)):
    oof = np.zeros(len(d))
    for f in range(5):
        tr, va = (d.fold != f).values, (d.fold == f).values
        for s in seeds:
            oof[va] += fit(d[tr], F, {**params, "random_state": s}).predict(d.loc[va, F]) / len(seeds)
    h = hit3(d, oof)
    fold_of = d.drop_duplicates("sample_id").set_index("sample_id").fold
    return oof, h, h.groupby(fold_of.reindex(h.index)).mean()


LOG = ROOT / "experiments" / "lgb_runs.csv"
HITS = {}   # nombre del experimento -> acierto por muestra


def experimento(nombre, groups, params=None, seeds=SEEDS_BARRIDO, nota=""):
    params = {**PARAMS, **(params or {})}
    t0 = time.time()
    d, F = build(lab, groups)
    oof, h, por_fold = cv(d, F, params, seeds)
    HITS[nombre] = h
    fila = dict(hora=time.strftime("%H:%M"), nombre=nombre, grupos="+".join(groups) or "-",
                params_cambiados=str({k: v for k, v in params.items() if PARAMS.get(k) != v}),
                n_features=len(F), cv_top3=round(h.mean(), 4), std_folds=round(por_fold.std(), 4),
                nota=f"{len(seeds)} semillas. {nota}".strip())
    pd.DataFrame([fila]).to_csv(LOG, mode="a", header=not LOG.exists(), index=False)
    print(f"{nombre:40s} CV top-3 = {h.mean():.4f}  (folds: {' '.join(f'{v:.3f}' for v in por_fold)})"
          f"  {len(F)} feats  {time.time() - t0:.0f}s", flush=True)
    return h.mean()

## 5. Barrido
**Ronda 1 (1 semilla, ya hecha):** base 0.6513 · +snapshot 0.6516 · +avion 0.6525 · +tipo 0.6542 · +tipo_estado 0.6549 · +contexto 0.6551 · todos juntos 0.6477. Conclusión: meterlo todo junto empeora; `snapshot` y `avion` no aportan.

**Ronda 2 (esta):**
1. **Cribado:** cada grupo candidato por separado sobre la base.
2. **Selección hacia delante:** se añaden los grupos que mejoraron, del mejor al peor, y cada uno **solo se queda si sube la CV al menos `MIN_GAIN`** respecto a lo que ya había.
3. **Parámetros** (solo con `RAPIDO = False`): sobre la mejor combinación, se prueban variantes de parámetros con el mismo criterio.

Con `RAPIDO = True` solo se criban los 4 grupos nuevos o prometedores (`contexto`, `exit_coord`, `alerta`, `deadline`): `tipo_estado` ya se midió con 2 semillas (0.6565 frente a 0.6570 de la base, no aporta) y `tipo` añade 14 columnas para +0.003 con 1 semilla.

`MIN_GAIN = 0.002`: por debajo lo consideramos ruido.

In [5]:
MIN_GAIN = 0.002
CANDIDATOS = (["contexto", "exit_coord", "alerta", "deadline"] if RAPIDO else
              ["tipo_estado", "contexto", "tipo", "exit_coord", "alerta", "deadline"])

# 1. Cribado
base_cv = experimento("base", [])
ganancia = {g: experimento(f"base+{g}", [g]) - base_cv for g in CANDIDATOS}

# 2. Selección hacia delante
mejor, mejor_cv, mejor_nombre = [], base_cv, "base"
for g in sorted((g for g in ganancia if ganancia[g] > 0), key=lambda g: -ganancia[g]):
    nombre = "base+" + "+".join(mejor + [g])
    cv_g = base_cv + ganancia[g] if not mejor else experimento(nombre, mejor + [g])
    if cv_g >= mejor_cv + MIN_GAIN:
        mejor, mejor_cv, mejor_nombre = mejor + [g], cv_g, nombre
        print(f"   -> se queda {g}: {mejor_cv:.4f}")
    else:
        print(f"   -> fuera {g}")
print(f"\nMejor combinación de features: {mejor or 'solo base'}  CV {mejor_cv:.4f}")

base                                     CV top-3 = 0.6513  (folds: 0.683 0.659 0.647 0.640 0.630)  88 feats  38s
base+contexto                            CV top-3 = 0.6551  (folds: 0.689 0.657 0.659 0.627 0.644)  94 feats  34s
base+exit_coord                          CV top-3 = 0.6506  (folds: 0.687 0.661 0.639 0.630 0.637)  96 feats  33s
base+alerta                              CV top-3 = 0.6571  (folds: 0.691 0.661 0.660 0.629 0.645)  93 feats  34s
base+deadline                            CV top-3 = 0.6510  (folds: 0.677 0.663 0.635 0.639 0.643)  92 feats  36s
   -> se queda alerta: 0.6571
base+alerta+contexto                     CV top-3 = 0.6596  (folds: 0.697 0.664 0.659 0.630 0.648)  99 feats  35s
   -> se queda contexto: 0.6596

Mejor combinación de features: ['alerta', 'contexto']  CV 0.6596


In [6]:
# 3. Parámetros sobre la mejor combinación
PARAM_CANDIDATOS = {
    "lento": dict(learning_rate=0.03, n_estimators=700),
    "regularizado": dict(min_child_samples=100, colsample_bytree=0.5, reg_lambda=5.0),
    "xendcg": dict(objective="rank_xendcg"),
}
mejores_params, base_params_cv = {}, mejor_cv
buenos = []
for nombre_p, p in ({} if RAPIDO else PARAM_CANDIDATOS).items():
    nombre = f"{mejor_nombre} [{nombre_p}]"
    cv_p = experimento(nombre, mejor, p)
    if cv_p >= base_params_cv + MIN_GAIN:
        buenos.append((cv_p, nombre_p, nombre))
if buenos:
    cv_p, nombre_p, nombre = max(buenos)
    mejores_params, mejor_cv, mejor_nombre = PARAM_CANDIDATOS[nombre_p], cv_p, nombre
    if len(buenos) > 1:   # probar a combinar los dos mejores
        (c1, n1, _), (c2, n2, _) = sorted(buenos, reverse=True)[:2]
        comb = {**PARAM_CANDIDATOS[n1], **PARAM_CANDIDATOS[n2]}
        nombre = f"{mejor_nombre.split(' [')[0]} [{n1}+{n2}]"
        cv_c = experimento(nombre, mejor, comb)
        if cv_c >= mejor_cv + MIN_GAIN:
            mejores_params, mejor_cv, mejor_nombre = comb, cv_c, nombre
print(f"\nGANADOR: {mejor_nombre}  CV {mejor_cv:.4f} ({len(SEEDS_BARRIDO)} semilla/s)  | params: {mejores_params or 'los de base'}")


GANADOR: base+alerta+contexto  CV 0.6596 (1 semilla/s)  | params: los de base


In [7]:
# Historial de experimentos de esta ronda
pd.read_csv(LOG).tail(len(HITS))

,hora,nombre,grupos,params_cambiados,n_features,cv_top3,std_folds,nota
14,19:09,base,-,{},88,0.6513,0.0204,1 semillas.
15,19:10,base+contexto,contexto,{},94,0.6551,0.0229,1 semillas.
16,19:10,base+exit_coord,exit_coord,{},96,0.6506,0.0232,1 semillas.
17,19:11,base+alerta,alerta,{},93,0.6571,0.0230,1 semillas.
18,19:11,base+deadline,deadline,{},92,0.6510,0.0177,1 semillas.
19,19:12,base+alerta+contexto,alerta+contexto,{},99,0.6596,0.0245,1 semillas.


## 6. Dónde falla
Acierto según **el tipo de la tarea correcta**, base frente al ganador. Útil para decidir qué atacar (y para Pau).

In [8]:
def por_tipo(h):
    y = lab[lab.target_index == 1].set_index("sample_id").task_index.map(lambda t: TYPES[t])
    return (pd.DataFrame({"tipo": y.reindex(h.index), "hit": h})
              .groupby("tipo").hit.agg(acierto="mean", muestras="size")
              .sort_values("muestras", ascending=False).round(3))

pd.concat({"base": por_tipo(HITS["base"]), "ganador": por_tipo(HITS[mejor_nombre])}, axis=1)

base          ganador         
                           acierto muestras acierto muestras
tipo                                                        
transfer_aircraft            0.855     3527   0.862     3527
assume_aircraft              0.827     3366   0.835     3366
zone_conflict                0.713     2121   0.714     2121
entry_coordination           0.353     1068   0.386     1068
return_to_route              0.513      937   0.529      937
exit_coordination            0.097      745   0.083      745
exit_conditions              0.194      448   0.196      448
conflict_resolution          0.315      390   0.341      390
qos_improvement              0.167      257   0.167      257
exit_conflict_resolution     0.142      197   0.157      197
non_conformance_resolution   0.640       86   0.628       86
entry_conflict_resolution    0.065       31   0.032       31
entry_conditions             0.500        2   0.500        2

## 7. Modelo final: guardar OOF, modelo y predicciones
Usa la combinación ganadora de la sección 5 (`BEST_GROUPS`, `BEST_PARAMS`) con `SEEDS_FINAL` semillas. Genera:
- `oof/lgb_adri.csv` — predicciones OOF (para el stacking de Juanma y el análisis de Pau)
- `models/lgb_adri_s<semilla>.txt` — **los pesos**: un modelo por semilla, entrenados con todo train+validation
- `models/lgb_adri_config.json` — grupos de features y parámetros usados
- `preds/lgb_adri.csv` y `submissions/sub_lgb_adri.csv` — predicciones de test (media de las semillas)

In [9]:
BEST_GROUPS, BEST_PARAMS = mejor, mejores_params   # para fijarlo a mano: BEST_GROUPS = [...]; BEST_PARAMS = {...}

params = {**PARAMS, **BEST_PARAMS}
d, F = build(lab, BEST_GROUPS)
oof, h, por_fold = cv(d, F, params, SEEDS_FINAL)
print(f"CV top-3 final ({len(SEEDS_FINAL)} semillas): {h.mean():.4f}  (folds: {' '.join(f'{v:.3f}' for v in por_fold)})")
pd.DataFrame([dict(hora=time.strftime("%H:%M"), nombre="FINAL " + mejor_nombre, grupos="+".join(BEST_GROUPS) or "-",
                   params_cambiados=str(BEST_PARAMS), n_features=len(F), cv_top3=round(h.mean(), 4),
                   std_folds=round(por_fold.std(), 4), nota=f"{len(SEEDS_FINAL)} semillas. Modelo guardado")]
             ).to_csv(LOG, mode="a", header=False, index=False)
d[["sample_id", "row_index"]].assign(score=oof).to_csv(ROOT / "oof" / f"{NAME}.csv", index=False)

dt, _ = build(test, BEST_GROUPS)
score = np.zeros(len(dt))
for s in SEEDS_FINAL:
    m = fit(d, F, {**params, "random_state": s})
    m.booster_.save_model(str(ROOT / "models" / f"{NAME}_s{s}.txt"))
    score += m.predict(dt[F]) / len(SEEDS_FINAL)
json.dump(dict(groups=BEST_GROUPS, params=params, seeds=SEEDS_FINAL, features=F, cv_top3=round(h.mean(), 4)),
          open(ROOT / "models" / f"{NAME}_config.json", "w"), indent=1)

pred = dt[["sample_id", "row_index"]].assign(score=score)
pred.to_csv(ROOT / "preds" / f"{NAME}.csv", index=False)
sub = (pred.sort_values(["sample_id", "score"], ascending=[True, False]).groupby("sample_id").row_index
           .apply(lambda r: " ".join(map(str, r.iloc[:3]))).rename("prediction").reset_index())
sub.to_csv(ROOT / "submissions" / f"sub_{NAME}.csv", index=False)
print("Guardado:", NAME, "| grupos:", BEST_GROUPS, "| test:", len(sub), "muestras")

CV top-3 final (2 semillas): 0.6569  (folds: 0.704 0.659 0.654 0.626 0.643)
Guardado: lgb_adri | grupos: ['alerta', 'contexto'] | test: 1863 muestras


### Comprobación con el scorer oficial
Con el OOF construimos un envío solo para las muestras de `validation.csv` y lo puntúa `Data/scorer.py`. Debe dar lo mismo que nuestro `hit3` en esas muestras.

In [10]:
val_ids = set(pd.read_csv(DATA / "validation.csv", usecols=["sample_id"]).sample_id)
o = d[["sample_id", "row_index"]].assign(score=oof)
o = o[o.sample_id.isin(val_ids)].sort_values(["sample_id", "score"], ascending=[True, False])
val_sub = o.groupby("sample_id").row_index.apply(lambda r: " ".join(map(str, r.iloc[:3]))).rename("prediction")
val_sub.reset_index().to_csv(ROOT / "submissions" / f"val_{NAME}.csv", index=False)
print(f"hit3 propio en validation: {h[h.index.isin(val_ids)].mean():.4f}")
r = subprocess.run([sys.executable, DATA / "scorer.py", "--labelled", DATA / "validation.csv",
                    ROOT / "submissions" / f"val_{NAME}.csv"], capture_output=True, text=True)
print("scorer oficial:", r.stdout or r.stderr)

hit3 propio en validation: 0.6765
scorer oficial: top-3 hit rate: 0.6765  (1252 samples)



### Cómo usan los demás el modelo guardado (sin entrenar)
Necesitan las funciones de la sección 3 (`build`) para crear las mismas features:
```python
cfg = json.load(open("models/lgb_adri_config.json"))
dt, F = build(test, cfg["groups"])
score = np.mean([lgb.Booster(model_file=f"models/lgb_adri_s{s}.txt").predict(dt[F]) for s in cfg["seeds"]], axis=0)
```